# 03 — XBRL Parsing & Auto-Labeling

Parses XBRL (inline and instance) documents to auto-generate training
labels. Each XBRL fact maps to an IFRS/GAAP taxonomy element, giving
us ground-truth `(line_item_text → taxonomy_code)` pairs for SBERT
training and zone classification labels.

**Input**: EDGAR 10-K HTML filings (inline XBRL) from notebook 01
**Output**: labeled_facts.json with taxonomy-tagged line items

In [ ]:
# Cell 1: Setup
!pip install -q arelle-release lxml tqdm PyYAML

import json, os, re
from pathlib import Path
from lxml import etree
from tqdm.auto import tqdm
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
EDGAR_DIR = DATA_DIR / 'edgar'
LABELS_DIR = DATA_DIR / 'labels'
LABELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'EDGAR dir: {EDGAR_DIR}')

In [ ]:
# Cell 2: Inline XBRL parser — extract tagged facts from HTML
XBRL_NS = {
    'ix': 'http://www.xbrl.org/2013/inlineXBRL',
    'us-gaap': 'http://fasb.org/us-gaap/2023',
    'ifrs-full': 'http://xbrl.ifrs.org/taxonomy/2023-03-23/ifrs-full',
    'dei': 'http://xbrl.sec.gov/dei/2023',
}

# Common IFRS/GAAP element → human-readable mapping
ELEMENT_LABELS = {
    'Revenue': 'Revenue',
    'Revenues': 'Revenue',
    'CostOfGoodsAndServicesSold': 'Cost of Revenue',
    'CostOfRevenue': 'Cost of Revenue',
    'GrossProfit': 'Gross Profit',
    'OperatingIncomeLoss': 'Operating Income',
    'NetIncomeLoss': 'Net Income',
    'EarningsPerShareBasic': 'Basic EPS',
    'Assets': 'Total Assets',
    'Liabilities': 'Total Liabilities',
    'StockholdersEquity': 'Total Equity',
    'CashAndCashEquivalentsAtCarryingValue': 'Cash & Equivalents',
    'NetCashProvidedByUsedInOperatingActivities': 'Operating Cash Flow',
}

def parse_inline_xbrl(html_path: Path) -> list[dict]:
    """Extract XBRL-tagged facts from an iXBRL HTML filing."""
    try:
        tree = etree.parse(str(html_path), etree.HTMLParser())
    except Exception as e:
        return []
    
    facts = []
    # Look for ix:nonFraction (numeric facts) and ix:nonNumeric
    for ns_prefix in ['ix']:
        ns_uri = XBRL_NS.get(ns_prefix, '')
        for tag in ['nonFraction', 'nonNumeric']:
            for elem in tree.iter(f'{{{ns_uri}}}{tag}'):
                name = elem.get('name', '')
                context = elem.get('contextRef', '')
                value = elem.text or ''
                if not name:
                    continue
                # Split namespace:element
                parts = name.split(':')
                namespace = parts[0] if len(parts) > 1 else ''
                element = parts[-1]
                facts.append({
                    'element': element,
                    'namespace': namespace,
                    'full_name': name,
                    'value': value.strip(),
                    'context': context,
                    'label': ELEMENT_LABELS.get(element, element),
                    'is_numeric': tag == 'nonFraction',
                })
    
    # Fallback: parse elements with us-gaap / ifrs-full namespaces directly
    for ns_prefix, ns_uri in [('us-gaap', XBRL_NS['us-gaap']), ('ifrs-full', XBRL_NS['ifrs-full'])]:
        for elem in tree.iter():
            if elem.tag and ns_uri in str(elem.tag):
                local = etree.QName(elem.tag).localname
                if local and local not in [f['element'] for f in facts]:
                    facts.append({
                        'element': local,
                        'namespace': ns_prefix,
                        'full_name': f'{ns_prefix}:{local}',
                        'value': (elem.text or '').strip(),
                        'context': '',
                        'label': ELEMENT_LABELS.get(local, local),
                        'is_numeric': False,
                    })
    return facts

# Test with one file
html_files = list(EDGAR_DIR.glob('*.html'))
if html_files:
    test_facts = parse_inline_xbrl(html_files[0])
    print(f'Parsed {len(test_facts)} facts from {html_files[0].name}')
    for f in test_facts[:10]:
        print(f"  {f['full_name']}: {f['value'][:50]}")
else:
    print('No HTML files found — run notebook 01 first')

In [ ]:
# Cell 3: Process all EDGAR filings
all_facts = []
file_stats = []

for html_path in tqdm(html_files, desc='Parsing iXBRL'):
    facts = parse_inline_xbrl(html_path)
    for f in facts:
        f['source_file'] = html_path.name
    all_facts.extend(facts)
    file_stats.append({'file': html_path.name, 'fact_count': len(facts)})

print(f'Total facts: {len(all_facts)} from {len(html_files)} filings')
print(f'Unique elements: {len(set(f["element"] for f in all_facts))}')

In [ ]:
# Cell 4: Build SBERT training pairs (line_item_text, taxonomy_code)
# Filter to high-value financial statement elements

FINANCIAL_ELEMENTS = {
    'Revenue', 'Revenues', 'CostOfGoodsAndServicesSold', 'CostOfRevenue',
    'GrossProfit', 'OperatingIncomeLoss', 'OperatingExpenses',
    'NetIncomeLoss', 'EarningsPerShareBasic', 'EarningsPerShareDiluted',
    'Assets', 'CurrentAssets', 'NoncurrentAssets', 'Liabilities',
    'CurrentLiabilities', 'NoncurrentLiabilities', 'StockholdersEquity',
    'CashAndCashEquivalentsAtCarryingValue', 'AccountsReceivableNetCurrent',
    'Inventory', 'PropertyPlantAndEquipmentNet', 'Goodwill',
    'LongTermDebt', 'ShortTermBorrowings',
    'NetCashProvidedByUsedInOperatingActivities',
    'NetCashProvidedByUsedInInvestingActivities',
    'NetCashProvidedByUsedInFinancingActivities',
    'DepreciationDepletionAndAmortization',
    'InterestExpense', 'IncomeTaxExpenseBenefit',
}

sbert_pairs = []
for fact in all_facts:
    if fact['element'] in FINANCIAL_ELEMENTS and fact['is_numeric']:
        sbert_pairs.append({
            'text': fact['label'],
            'taxonomy_code': fact['full_name'],
            'element': fact['element'],
            'namespace': fact['namespace'],
            'source': fact.get('source_file', ''),
        })

# Deduplicate by (text, taxonomy_code)
seen = set()
unique_pairs = []
for p in sbert_pairs:
    key = (p['text'], p['taxonomy_code'])
    if key not in seen:
        seen.add(key)
        unique_pairs.append(p)

print(f'SBERT training pairs: {len(unique_pairs)} (from {len(sbert_pairs)} total)')

In [ ]:
# Cell 5: Save labeled datasets
# Full facts
with open(LABELS_DIR / 'all_xbrl_facts.json', 'w') as f:
    json.dump(all_facts, f, indent=2)

# SBERT pairs
with open(LABELS_DIR / 'sbert_training_pairs.json', 'w') as f:
    json.dump(unique_pairs, f, indent=2)

# Zone labeling: classify facts into statement types
ZONE_MAPPING = {
    'income_statement': ['Revenue', 'CostOfRevenue', 'GrossProfit', 'OperatingIncomeLoss',
                         'NetIncomeLoss', 'EarningsPerShareBasic', 'OperatingExpenses',
                         'InterestExpense', 'IncomeTaxExpenseBenefit'],
    'balance_sheet': ['Assets', 'CurrentAssets', 'NoncurrentAssets', 'Liabilities',
                      'StockholdersEquity', 'CashAndCashEquivalentsAtCarryingValue',
                      'Inventory', 'Goodwill', 'PropertyPlantAndEquipmentNet'],
    'cash_flow': ['NetCashProvidedByUsedInOperatingActivities',
                  'NetCashProvidedByUsedInInvestingActivities',
                  'NetCashProvidedByUsedInFinancingActivities',
                  'DepreciationDepletionAndAmortization'],
}

zone_labels = []
for fact in all_facts:
    for zone, elements in ZONE_MAPPING.items():
        if fact['element'] in elements:
            zone_labels.append({'text': fact['label'], 'zone': zone, 'element': fact['element']})
            break

with open(LABELS_DIR / 'zone_labels.json', 'w') as f:
    json.dump(zone_labels, f, indent=2)

print(f'Saved:')
print(f'  all_xbrl_facts.json: {len(all_facts)} facts')
print(f'  sbert_training_pairs.json: {len(unique_pairs)} pairs')
print(f'  zone_labels.json: {len(zone_labels)} zone labels')